# B5–B6: Deploying, Operating and Observing Payment-Matching Agents

This lab defines a production deployment contract, emits structured traces for an end-to-end payment match, and calculates operational SLO metrics.

In [1]:
# B5: these settings map directly to Container Apps / Kubernetes environment variables.
DEPLOYMENT_CONTRACT = {
    'service_name': 'payment-matching-agent',
    'image': 'contoso/payment-matching-agent:1.0.0',
    'replicas': {'min': 1, 'max': 5},
    'health_endpoint': '/healthz',
    'environment': ['AGENT_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'APPLICATIONINSIGHTS_CONNECTION_STRING'],
    'release_gate': {'evaluation_pass_rate': '>= 95%', 'p95_latency_ms': '<= 5000', 'critical_policy_failures': 0},
}

def health_check(configuration):
    assert configuration['replicas']['min'] >= 1
    assert '/health' in configuration['health_endpoint']
    assert 'AZURE_OPENAI_API_KEY' in configuration['environment']
    return {'status': 'healthy', 'service': configuration['service_name']}

print('Deployment contract:', DEPLOYMENT_CONTRACT)
print('Health check:', health_check(DEPLOYMENT_CONTRACT))

Deployment contract: {'service_name': 'payment-matching-agent', 'image': 'contoso/payment-matching-agent:1.0.0', 'replicas': {'min': 1, 'max': 5}, 'health_endpoint': '/healthz', 'environment': ['AGENT_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'APPLICATIONINSIGHTS_CONNECTION_STRING'], 'release_gate': {'evaluation_pass_rate': '>= 95%', 'p95_latency_ms': '<= 5000', 'critical_policy_failures': 0}}
Health check: {'status': 'healthy', 'service': 'payment-matching-agent'}


In [2]:
# B6: structured spans are the payload you would send to Application Insights / OpenTelemetry.
from datetime import datetime, timezone
from statistics import quantiles

trace_id = 'trace-pay-78'
spans = [
    {'name': 'ingest_remittance', 'latency_ms': 22, 'status': 'ok', 'payment_id': 'PAY-78'},
    {'name': 'two_way_match', 'latency_ms': 8, 'status': 'no_match', 'payment_id': 'PAY-78'},
    {'name': 'smart_match', 'latency_ms': 55, 'status': 'candidate_found', 'confidence': 0.97},
    {'name': 'human_approval_gate', 'latency_ms': 4, 'status': 'required'},
]
for span in spans:
    span.update({'trace_id': trace_id, 'timestamp_utc': datetime.now(timezone.utc).isoformat()})

latencies = [span['latency_ms'] for span in spans]
success_rate = sum(span['status'] != 'error' for span in spans) / len(spans)
p95 = quantiles(latencies, n=20, method='inclusive')[18]

for span in spans:
    print(span)
print(f'\nTrace: {trace_id} | success rate: {success_rate:.0%} | p95 latency: {p95:.1f} ms')
assert success_rate == 1.0
assert p95 < 5000

{'name': 'ingest_remittance', 'latency_ms': 22, 'status': 'ok', 'payment_id': 'PAY-78', 'trace_id': 'trace-pay-78', 'timestamp_utc': '2026-08-18T02:41:53.279095+00:00'}
{'name': 'two_way_match', 'latency_ms': 8, 'status': 'no_match', 'payment_id': 'PAY-78', 'trace_id': 'trace-pay-78', 'timestamp_utc': '2026-08-18T02:41:53.279095+00:00'}
{'name': 'smart_match', 'latency_ms': 55, 'status': 'candidate_found', 'confidence': 0.97, 'trace_id': 'trace-pay-78', 'timestamp_utc': '2026-08-18T02:41:53.279095+00:00'}
{'name': 'human_approval_gate', 'latency_ms': 4, 'status': 'required', 'trace_id': 'trace-pay-78', 'timestamp_utc': '2026-08-18T02:41:53.279095+00:00'}

Trace: trace-pay-78 | success rate: 100% | p95 latency: 50.0 ms
